# Load Data

In [0]:
txn = spark.read.table("workspace.refined_upi.transactions_silver")
settlements = spark.read.table("workspace.raw_upi.bank_settlements_bronze")

# Daily_txn_summary

In [0]:
from pyspark.sql.functions import *
daily_txn_summary = txn.withColumn("txn_date", to_date("transaction_timestamp_ist")) \
    .groupBy("txn_date", "state", "upi_app") \
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("status") == "SUCCESS", 1).otherwise(0)).alias("successful_txns"),
        sum("amount").alias("total_amount"),
        avg("amount").alias("avg_ticket_size")
    )

In [0]:
daily_txn_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.consolidated_upi.daily_txn_summary")

spark.sql("""
OPTIMIZE workspace.consolidated_upi.daily_txn_summary
ZORDER BY (txn_date, state, upi_app)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
%sql
-- SELECT * FROM workspace.consolidated_upi.daily_txn_summary WHERE txn_date = '2024-03-29';

txn_date,state,upi_app,total_transactions,successful_txns,total_amount,avg_ticket_size
2024-03-29,West Bengal,PhonePe,1,1,null,null
2024-03-29,Maharashtra,BHIM,1,1,812.62,812.62
2024-03-29,Maharashtra,Amazon Pay,2,0,null,null
2024-03-29,Telangana,Paytm,1,0,null,null
2024-03-29,Pune,BHIM,2,2,null,null
2024-03-29,Pune,Paytm,1,0,null,null
2024-03-29,Karnataka,GPay,1,1,256.5,256.5
2024-03-29,Gujarat,PhonePe,5,4,2364.12,591.03
2024-03-29,Telangana,BHIM,2,1,832.95,832.95
2024-03-29,Rajasthan,Paytm,1,1,990.76,990.76


# Merchant_performance

In [0]:
merchant_performance = txn.groupBy("merchant_id", "merchant_category") \
    .agg(
        count("*").alias("total_txns"),
        sum(when(col("status") == "SUCCESS", 1).otherwise(0)).alias("success_txns"),
        avg("amount").alias("avg_ticket_size"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_txns")
    ) \
    .withColumn("success_rate", col("success_txns") / col("total_txns")) \
    .withColumn("fraud_rate", col("fraud_txns") / col("total_txns"))

In [0]:
merchant_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.consolidated_upi.merchant_performance")

spark.sql("""
OPTIMIZE workspace.consolidated_upi.merchant_performance
ZORDER BY (merchant_id, merchant_category)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
%sql
-- SELECT * FROM workspace.consolidated_upi.merchant_performance WHERE merchant_category = "Sporting Goods"

merchant_id,merchant_category,total_txns,success_txns,avg_ticket_size,fraud_txns,success_rate,fraud_rate
MER00063,Sporting Goods,3,1,null,1,0.3333333333333333,0.3333333333333333
MER00015,Sporting Goods,5,4,322.1266666666666,1,0.8,0.2
MER00382,Sporting Goods,4,3,297.92,0,0.75,0.0
MER00336,Sporting Goods,2,0,null,0,0.0,0.0
MER00181,Sporting Goods,3,0,399.745,2,0.0,0.6666666666666666
MER00135,Sporting Goods,3,2,348.3733333333333,1,0.6666666666666666,0.3333333333333333
MER00439,Sporting Goods,3,2,60.15,1,0.6666666666666666,0.3333333333333333
MER00164,Sporting Goods,5,2,566.6,0,0.4,0.0
MER00241,Sporting Goods,1,0,null,0,0.0,0.0
MER00497,Sporting Goods,9,4,196.0666666666667,5,0.4444444444444444,0.5555555555555556


# Settlement_reconciliation

In [0]:
from pyspark.sql.functions import *

# Renaming
txn = txn \
    .withColumnRenamed("_batch_id", "txn_batch_id") \
    .withColumnRenamed("_file_name", "txn_file_name") \
    .withColumnRenamed("_ingest_timestamp", "txn_ingest_timestamp") \
    .withColumnRenamed("_source_system", "txn_source_system") \
    .withColumnRenamed("bank_reference_number", "txn_bank_reference_number") \
    .withColumnRenamed("merchant_id", "txn_merchant_id")



settlements = settlements \
    .withColumnRenamed("_batch_id", "settlement_batch_id") \
    .withColumnRenamed("_file_name", "settlement_file_name") \
    .withColumnRenamed("_ingest_timestamp", "settlement_ingest_timestamp") \
    .withColumnRenamed("_source_system", "settlement_source_system") \
    .withColumnRenamed("bank_reference_number", "settlement_bank_reference_number") \
    .withColumnRenamed("merchant_id", "settlement_merchant_id")


# join

joined_df = txn.join(
    settlements,
    txn.txn_bank_reference_number == settlements.settlement_bank_reference_number,
    "left"
)

# reconciliation logic
settlement_reconciliation = joined_df \
.withColumn(
    "is_reconciled",
    when(col("settlement_status") == "SUCCESS", True).otherwise(False)
) \
.withColumn(
    "reconciliation_status",
    when(col("settlement_status").isNull(), "NOT_FOUND")
    .when(col("settlement_status") == "SUCCESS", "RECONCILED")
    .otherwise(col("settlement_status"))
) \
.withColumn(
    "amount_match_flag",
    when(col("amount") == col("net_settlement_amount"), True).otherwise(False)
)



settlement_reconciliation_final = settlement_reconciliation.select(
    "transaction_id",
    "transaction_timestamp",
    "sender_vpa",
    "receiver_vpa",
    "receiver_merchant_id",
    "upi_app",
    "transaction_type",
    "amount",
    "status",
    "failure_reason",
    "state",
    "is_repeat_txn",

    "txn_bank_reference_number",
    "settlement_bank_reference_number",

    "txn_merchant_id",
    "settlement_merchant_id",

    "settlement_id",
    "settlement_date",
    "value_date",
    "bank_name",
    "gross_amount",
    "charges_deducted",
    "net_settlement_amount",
    "settlement_status",

    "reconciliation_status",
    "is_reconciled",
    "amount_match_flag",

    "txn_ingest_timestamp",
    "settlement_ingest_timestamp",
    "txn_source_system",
    "settlement_source_system"
)

settlement_reconciliation_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.consolidated_upi.settlement_reconciliation")


spark.sql("""
OPTIMIZE workspace.consolidated_upi.settlement_reconciliation
ZORDER BY (transaction_id, settlement_status)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

# Fraud_analytics

In [0]:

fraud_analytics = txn.withColumn("txn_date", to_date("transaction_timestamp_ist")) \
    .withColumn(
        "loss_amount",
        when(col("is_fraud") == True, col("amount")).otherwise(0)
    ) \
    .groupBy("fraud_rule", "merchant_category", "txn_date") \
    .agg(
        count("*").alias("total_txns"),
        sum("loss_amount").alias("total_loss"),
        avg("risk_score").alias("avg_risk_score"),
        sum(col("is_fraud").cast("int")).alias("fraud_count")
    )
fraud_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.consolidated_upi.fraud_analytics")

spark.sql("""
OPTIMIZE workspace.consolidated_upi.fraud_analytics
ZORDER BY (fraud_rule, merchant_category)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
%sql
-- SELECT * FROM workspace.consolidated_upi.fraud_analytics;

fraud_rule,merchant_category,txn_date,total_txns,total_loss,avg_risk_score,fraud_count
Amount anomaly � 10x avg ticket size,Transportation,2024-03-30,1,null,0.58,1
OTP bypass attempt,null,2024-03-16,1,455.23,0.75,1
Round-amount structuring,Drug Stores / Pharmacies,2024-02-17,1,174.18,0.63,1
Blacklisted VPA,Transportation,2024-01-01,1,458.39,0.83,1
Round-amount structuring,Hotels and Lodging,2024-01-27,1,496.68,0.88,1
Cross-border attempt,null,2024-02-14,3,171.71,0.6233333333333334,3
High velocity � >5 txn in 10 min,Transportation,2024-02-28,1,80.95,0.73,1
High velocity � >5 txn in 10 min,null,2024-01-27,1,799.49,0.65,1
Amount anomaly � 10x avg ticket size,null,2024-03-12,1,null,0.63,1
OTP bypass attempt,null,2024-02-19,1,null,0.86,1
